# SmartOps dashboard: train and export the model (Google Colab)

Run this notebook **top to bottom in Google Colab** (Runtime > Run all). It needs no local install and no GitHub login: it downloads the public dataset (MIT license) from Hugging Face.

**What it does**
1. Loads the project's official splits: train + val = 16,604 development rows, test = 2,931 held-out rows.
2. Fits the model recommended by the evaluation work, a **spline logistic regression** (6 quantile knots, cubic, linear extrapolation, `StandardScaler`, `C = 0.03`) **on development data only**. The test set is scored once, for reporting, and never used to fit or tune anything.
3. Computes cross-validated metrics (same protocol as `src/common.py`: repeated stratified 5-fold x 3, seed 42), out-of-fold decision tables for the threshold and cost views, and reference predictions.
4. Exports everything the dashboard needs as `model.json` and `model_data.js`, then **checks the export** by re-computing probabilities from the exported numbers alone.

**After it finishes:** download `model.json` and `model_data.js` and put them in `dashboard/model/` in the repository (replace the existing files), then run the dashboard tests (`node --test dashboard/tests/core.test.js`).

Why this model and not the penalised GAM (20 splines)? Test AUC is 0.7099 vs 0.7095 (a difference of -0.0004, 95% CI -0.002 to +0.001, so they are equivalent), and the spline logistic regression is a plain linear model on spline features, so its predictions can be reproduced exactly in a browser with a few hundred numbers.

In [ ]:
import json, math, urllib.request, warnings
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
warnings.filterwarnings('ignore')
print('scikit-learn', sklearn.__version__, '| numpy', np.__version__)

TARGET = 'Engine Condition'
FEATURES = ['Engine RPM', 'Lub Oil Pressure', 'Fuel Pressure', 'Coolant Pressure', 'Lub Oil Temp', 'Coolant Temp']
SEED = 42
BASE = 'https://huggingface.co/datasets/jskswamy/predictive-maintenance-data/resolve/main/'
DATA = Path('data'); DATA.mkdir(exist_ok=True)
for name in ['train.csv', 'val.csv', 'test.csv']:
    if not (DATA / name).exists():
        urllib.request.urlretrieve(BASE + name, DATA / name)

train, val, test = (pd.read_csv(DATA / f) for f in ['train.csv', 'val.csv', 'test.csv'])
dev = pd.concat([train, val], ignore_index=True)
Xd, yd = dev[FEATURES].copy(), dev[TARGET].copy()
Xt, yt = test[FEATURES].copy(), test[TARGET].copy()
print('dev', Xd.shape, 'test', Xt.shape, '| prevalence dev', round(yd.mean(), 4), 'test', round(yt.mean(), 4))
assert (len(Xd), len(Xt)) == (16604, 2931), 'unexpected split sizes'

## 1. The model

Same definition as `spline_lr` in `experiments/models.py`, with the `C` chosen in the project's evaluation (6 knots, C = 0.03).

In [ ]:
def spline_lr(n_knots=6, degree=3, C=0.03):
    return make_pipeline(
        SplineTransformer(n_knots=n_knots, degree=degree, knots='quantile', extrapolation='linear'),
        StandardScaler(),
        LogisticRegression(C=C, max_iter=5000),
    )

model = spline_lr()

# Cross-validated metrics on development data (shared protocol)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
res = cross_validate(clone(model), Xd, yd, cv=cv, n_jobs=-1,
                     scoring={'auc': 'roc_auc', 'pr_auc': 'average_precision', 'brier': 'neg_brier_score'})
cv_metrics = {
    'n_folds': 15,
    'auc_mean': float(res['test_auc'].mean()), 'auc_std': float(res['test_auc'].std()),
    'pr_auc_mean': float(res['test_pr_auc'].mean()), 'pr_auc_std': float(res['test_pr_auc'].std()),
    'brier_mean': float((-res['test_brier']).mean()), 'brier_std': float((-res['test_brier']).std()),
}
print(cv_metrics)

# Final fit: DEVELOPMENT DATA ONLY
model.fit(Xd, yd)

## 2. Test set (scored once, for reporting only)

In [ ]:
p_test = model.predict_proba(Xt)[:, 1]
test_auc = roc_auc_score(yt, p_test)
rng = np.random.default_rng(SEED)
yt_a, pt_a = yt.to_numpy(), p_test
boots = []
for _ in range(2000):
    idx = rng.integers(0, len(yt_a), len(yt_a))
    if len(np.unique(yt_a[idx])) == 2:
        boots.append(roc_auc_score(yt_a[idx], pt_a[idx]))
ci = [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))]
test_metrics = {
    'n': int(len(yt)), 'prevalence': float(yt.mean()), 'auc': float(test_auc), 'auc_ci': ci,
    'pr_auc': float(average_precision_score(yt, p_test)), 'brier': float(brier_score_loss(yt, p_test)),
    'brier_prevalence_baseline': float(brier_score_loss(yt, np.full(len(yt), yt.mean()))),
}
print(test_metrics)
# The project report gives test AUC 0.7099 (95% CI 0.692 to 0.729) for this model.
if abs(test_metrics['auc'] - 0.7099) > 0.0005:
    print('WARNING: test AUC differs from the project report (0.7099). Check library versions before using this export.')

## 3. Out-of-fold decision tables (development data)

Every development row is scored by a model that never saw it (one stratified 5-fold pass). The dashboard uses these tables for the threshold, capacity and cost views, so no number there comes from the test set or from in-sample fits.

In [ ]:
oof = cross_val_predict(clone(model), Xd, yd, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                        method='predict_proba', n_jobs=-1)[:, 1]
y_a = yd.to_numpy()

def table(p, y, grid):
    out = {'tp': [], 'fp': [], 'fn': [], 'tn': []}
    for t in grid:
        f = p >= t
        out['tp'].append(int((f & (y == 1)).sum())); out['fp'].append(int((f & (y == 0)).sum()))
        out['fn'].append(int((~f & (y == 1)).sum())); out['tn'].append(int((~f & (y == 0)).sum()))
    return out

grid = [round(i / 100, 2) for i in range(0, 101)]
oof_tab = table(oof, y_a, grid); test_tab = table(p_test, yt_a, grid)

order = np.argsort(-oof, kind='stable')
cum_pos = np.cumsum(y_a[order])
pct = list(range(1, 101))
n_at = [int(math.ceil(k / 100 * len(oof))) for k in pct]
capture = {'pct': pct, 'n': n_at, 'tp': [int(cum_pos[n - 1]) for n in n_at]}

bins = np.linspace(0, 1, 11)
cal = []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (oof >= lo) & ((oof < hi) if hi < 1 else (oof <= hi))
    if m.sum():
        cal.append({'lo': float(lo), 'hi': float(hi), 'n': int(m.sum()),
                    'mean_pred': float(oof[m].mean()), 'observed': float(y_a[m].mean())})

oof_auc = float(roc_auc_score(y_a, oof))
print('OOF AUC', round(oof_auc, 4), '| rows', len(oof))

## 4. Export the fitted model

Everything is read from the fitted scikit-learn objects: spline knots, scaler, logistic coefficients.

In [ ]:
spl = model.named_steps['splinetransformer']
scaler = model.named_steps['standardscaler']
lr = model.named_steps['logisticregression']
n_feat = len(FEATURES)
n_splines = spl.n_features_out_ // n_feat
degree = spl.degree
knots = [[float(v) for v in b.t] for b in spl.bsplines_]
coef = lr.coef_[0]
mean, scale = scaler.mean_, scaler.scale_

# Per-feature centred log-odds contribution: intercept + sum_j contribution_j(x_j) == logit
Zd = spl.transform(Xd)
importance = {}
for j, f in enumerate(FEATURES):
    sl = slice(j * n_splines, (j + 1) * n_splines)
    contrib = (Zd[:, sl] - mean[sl]) @ (coef[sl] / scale[sl])
    importance[f] = {'mean_abs_logit': float(np.abs(contrib).mean()), 'sd_logit': float(contrib.std())}

ranges = {f: {'min': float(Xd[f].min()), 'p01': float(Xd[f].quantile(.01)), 'p25': float(Xd[f].quantile(.25)),
              'median': float(Xd[f].median()), 'p75': float(Xd[f].quantile(.75)),
              'p99': float(Xd[f].quantile(.99)), 'max': float(Xd[f].max()), 'mean': float(Xd[f].mean())}
          for f in FEATURES}

# Reference rows for parity checks: 250 real test rows + edge cases (outside the training range, on knots)
ref_rng = np.random.default_rng(SEED)
real = Xt.sample(250, random_state=SEED).to_numpy()
med = Xd.median().to_numpy()
edge = []
for j, f in enumerate(FEATURES):
    lo, hi = Xd[f].min(), Xd[f].max(); span = hi - lo
    base_knots = knots[j][degree: len(knots[j]) - degree]
    for v in [lo - 0.5 * span, lo - 1e-6, lo, hi, hi + 1e-6, hi + 0.5 * span] + list(base_knots):
        row = med.copy(); row[j] = v; edge.append(row)
ref_X = np.vstack([real, np.array(edge)])
ref_p = model.predict_proba(pd.DataFrame(ref_X, columns=FEATURES))[:, 1]

artifact = {
    'meta': {
        'name': 'SmartOps spline logistic regression',
        'description': 'Additive model: cubic B-spline basis per sensor (6 quantile knots, linear extrapolation), standardised, L2 logistic regression (C = 0.03).',
        'target': 'Engine Condition = 1 (Needs Maintenance), a dataset label with unknown origin',
        'trained_on': 'train + val only (development set)',
        'n_dev': int(len(Xd)), 'created': date.today().isoformat(),
        'sklearn_version': sklearn.__version__,
        'source': 'https://huggingface.co/datasets/jskswamy/predictive-maintenance-data (MIT license per dataset card)',
        'notebook': 'notebooks/dashboard_model_export.ipynb',
    },
    'features': FEATURES,
    'units': ['rpm', 'bar', 'bar', 'bar', 'degC', 'degC'],
    'spline': {'degree': int(degree), 'n_splines': int(n_splines), 'extrapolation': 'linear', 'knots': knots},
    'scaler': {'mean': [float(v) for v in mean], 'scale': [float(v) for v in scale]},
    'logistic': {'coef': [float(v) for v in coef], 'intercept': float(lr.intercept_[0])},
    'ranges': ranges,
    'importance': importance,
    'metrics': {'cv': cv_metrics, 'test': test_metrics, 'oof_auc': oof_auc,
                'dev_prevalence': float(yd.mean()), 'n_dev': int(len(yd))},
    'oof': {'n': int(len(oof)), 'prevalence': float(y_a.mean()), 'thresholds': grid, **oof_tab,
            'score_quantiles': [float(v) for v in np.quantile(oof, np.linspace(0, 1, 101))]},
    'test_table': {'n': int(len(yt)), 'thresholds': grid, **test_tab},
    'capture': capture,
    'calibration': cal,
    'reference': {'X': ref_X.tolist(), 'proba': [float(v) for v in ref_p], 'n_real_test_rows': 250},
}
out = Path('model_out'); out.mkdir(exist_ok=True)
text = json.dumps(artifact)
(out / 'model.json').write_text(text)
(out / 'model_data.js').write_text('window.SMARTOPS_MODEL = ' + text + ';\n')
print('wrote', [(p.name, round(p.stat().st_size / 1024, 1), 'KB') for p in out.iterdir()])

## 5. Check the export

Re-compute every reference probability from the exported numbers only (a from-scratch NumPy re-implementation of the B-spline basis, standardisation and logistic function) and compare with scikit-learn. The dashboard's JavaScript uses the same maths and is tested against the same reference rows.

In [ ]:
from scipy.interpolate import BSpline

def basis_row(x, t, k, n_basis):
    # evaluate all B-spline basis functions at x, with sklearn's linear extrapolation outside the base interval
    t = np.asarray(t); lo, hi = t[k], t[len(t) - k - 1]
    def ev(v, nu=0):
        out = np.zeros(n_basis)
        for i in range(n_basis):
            c = np.zeros(n_basis); c[i] = 1.0
            out[i] = BSpline(t, c, k, extrapolate=False)(v, nu=nu) if lo <= v <= hi else 0.0
        return out
    if x < lo:
        return ev(lo) + (x - lo) * ev(lo, 1)
    if x > hi:
        return ev(hi) + (x - hi) * ev(hi, 1)
    return ev(x)

def predict_from_artifact(a, X):
    k, nb = a['spline']['degree'], a['spline']['n_splines']
    mu, sd = np.array(a['scaler']['mean']), np.array(a['scaler']['scale'])
    w, b = np.array(a['logistic']['coef']), a['logistic']['intercept']
    out = []
    for row in X:
        z = np.concatenate([basis_row(row[j], a['spline']['knots'][j], k, nb) for j in range(len(row))])
        out.append(1 / (1 + np.exp(-(((z - mu) / sd) @ w + b))))
    return np.array(out)

chk = predict_from_artifact(json.loads(text), ref_X)
diff = np.abs(chk - ref_p).max()
print('max |from-scratch - sklearn| over', len(ref_p), 'reference rows =', diff)
assert diff < 1e-8, 'export does not reproduce scikit-learn predictions'
print('OK: export reproduces scikit-learn.')

In [ ]:
from google.colab import files
files.download('model_out/model.json')
files.download('model_out/model_data.js')

## Next

Copy both files into `dashboard/model/`, run `node --test dashboard/tests/core.test.js` (checks the JavaScript against the reference rows above) and open `dashboard/index.html`.